# 01 — IGC Track Exploration

Exploratory analysis of imported IGC flight tracks from Eagle Ridge Flying Site.

**Goals:**
- Load IGC files and inspect GPS fix quality
- Visualise altitude profiles and vario distributions
- Identify climb, glide, and sink segments
- Map hotspot density around the South Bowl and Eagle Ridge Main

**Advisory**: This notebook is for research and data exploration only. Not for operational flight planning.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

from data_ingestion.flights.igc_parser import IGCParser, FlightSegmentType

print('IGC exploration notebook ready.')

## 1. Load IGC Files

In [ ]:
# Set path to your IGC files
IGC_DIR = Path('../data/igc')  # adjust as needed

parser = IGCParser()
all_flights = []

if IGC_DIR.exists():
    for igc_file in sorted(IGC_DIR.glob('*.igc')):
        try:
            flight = parser.parse(igc_file)
            all_flights.append(flight)
            print(f'{igc_file.name}: {len(flight.fixes)} fixes, {len(flight.segments)} segments')
        except Exception as e:
            print(f'{igc_file.name}: ERROR — {e}')
else:
    print(f'Directory not found: {IGC_DIR}')
    print('To import tracks: uv run python scripts/import_igc.py --dir /path/to/igc_files')

print(f'\nLoaded {len(all_flights)} flights.')

## 2. Altitude Profile

In [ ]:
if not all_flights:
    print('No flights loaded — skipping altitude plot.')
else:
    fig, axes = plt.subplots(min(len(all_flights), 4), 1, figsize=(12, 3 * min(len(all_flights), 4)))
    if len(all_flights) == 1:
        axes = [axes]

    for i, (ax, flight) in enumerate(zip(axes, all_flights[:4])):
        times = [f.time_utc for f in flight.fixes]
        alts = [f.press_alt_m for f in flight.fixes]
        
        ax.plot(times, alts, linewidth=0.8, color='#2563eb')
        ax.axhline(1340, color='#94a3b8', linestyle='--', linewidth=0.6, label='Launch elevation (1340m)')
        ax.set_ylabel('Altitude (m AMSL)')
        ax.set_title(f'Flight {i+1}: {len(flight.fixes)} fixes')
        ax.legend(fontsize=8)
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}m'))

    plt.tight_layout()
    plt.suptitle('Eagle Ridge — Altitude Profiles', y=1.01, fontsize=12, fontweight='bold')
    plt.show()

## 3. Vario Distribution

In [ ]:
if all_flights:
    all_varios = []
    for flight in all_flights:
        all_varios.extend([f.vario_ms for f in flight.fixes if hasattr(f, 'vario_ms') and f.vario_ms is not None])

    if all_varios:
        vario_arr = np.array(all_varios)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.hist(vario_arr, bins=80, color='#2563eb', alpha=0.7, edgecolor='white', linewidth=0.3)
        ax.axvline(0.3, color='#10b981', linestyle='--', label='Climb threshold (0.3 m/s)')
        ax.axvline(-0.8, color='#ef4444', linestyle='--', label='Sink threshold (-0.8 m/s)')
        ax.set_xlabel('Vertical speed (m/s)')
        ax.set_ylabel('Count')
        ax.set_title('Vario Distribution — All Eagle Ridge Tracks')
        ax.legend()
        ax.set_xlim(-6, 6)
        plt.tight_layout()
        plt.show()
        
        print(f'N fixes with vario: {len(vario_arr)}')
        print(f'Mean: {vario_arr.mean():.2f} m/s | Median: {np.median(vario_arr):.2f} m/s')
        print(f'Climb fraction: {(vario_arr >= 0.3).mean() * 100:.1f}%')
        print(f'Sink fraction:  {(vario_arr <= -0.8).mean() * 100:.1f}%')
    else:
        print('No vario data computed — ensure parser computes vario.')

## 4. Segment Statistics

In [ ]:
if all_flights:
    rows = []
    for i, flight in enumerate(all_flights):
        for seg in flight.segments:
            rows.append({
                'flight_idx': i,
                'segment_type': seg.segment_type.value if hasattr(seg.segment_type, 'value') else seg.segment_type,
                'duration_s': seg.duration_s,
                'altitude_gain_m': seg.altitude_gain_m,
                'avg_vario_ms': seg.avg_vario_ms,
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        print('Segment statistics by type:')
        display(df.groupby('segment_type').agg(
            count=('duration_s', 'count'),
            total_time_min=('duration_s', lambda x: x.sum() / 60),
            avg_duration_s=('duration_s', 'mean'),
            avg_vario=('avg_vario_ms', 'mean'),
            max_vario=('avg_vario_ms', 'max'),
        ).round(2))
    else:
        print('No segments found.')

## 5. Spatial Climb Density Map

In [ ]:
# Eagle Ridge bounding box
LAT_MIN, LAT_MAX = 35.47, 35.51
LON_MIN, LON_MAX = -118.21, -118.16

if all_flights:
    climb_lats, climb_lons = [], []
    for flight in all_flights:
        for fix in flight.fixes:
            if hasattr(fix, 'vario_ms') and fix.vario_ms is not None and fix.vario_ms >= 0.3:
                if LAT_MIN <= fix.lat <= LAT_MAX and LON_MIN <= fix.lon <= LON_MAX:
                    climb_lats.append(fix.lat)
                    climb_lons.append(fix.lon)

    fig, ax = plt.subplots(figsize=(10, 8))

    if climb_lats:
        h = ax.hist2d(climb_lons, climb_lats, bins=40,
                      range=[[LON_MIN, LON_MAX], [LAT_MIN, LAT_MAX]],
                      cmap='hot_r', density=True)
        plt.colorbar(h[3], ax=ax, label='Climb density (normalised)')

    # Mark site features
    ax.plot(-118.187, 35.492, 'b^', markersize=10, label='Eagle Ridge Main Launch', zorder=5)
    ax.plot(-118.192, 35.478, 'gs', markersize=8, label='Valley LZ', zorder=5)
    ax.plot(-118.188, 35.487, 'ro', markersize=8, label='South Bowl Trigger', zorder=5)

    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Eagle Ridge — Climb Density Map')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'Total climb fixes plotted: {len(climb_lats)}')
else:
    print('No flights loaded.')

## 6. Summary

Key findings from this exploration:
- (Fill in after running with real data)
- South Bowl thermal trigger location
- Peak vario distribution
- Segment type proportions

Next: `02_weather_analysis.ipynb`